In [1]:
# Imports (todos nesta célula)
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "repeat_purchase_90d"
GROUP = "customer_id"
DATE = "purchase_date"
OUT_OF_MODEL = [GROUP, DATE]   # ficam na tabela, mas nunca entram no modelo
METRIC = "f1"                  # métrica que decide as comparações (accuracy ao lado)

In [2]:
# ===== [2.0] BASE COMUM - início =====
# Passo 1 - dados de partida, ordenados por data como na W5 (secção 3.4).
# As linhas sem data não têm lugar na cronologia, por isso ficam de fora.
raw = pd.read_csv("../data/train.csv", index_col="ID")

known_time = raw[DATE].notna()
# kind="stable" mantém pela ordem do ficheiro as compras do mesmo dia (W5)
time_order = raw.loc[known_time, DATE].sort_values(kind="stable").index
frame = raw.loc[time_order]

print("linhas excluídas por não terem data:", int((~known_time).sum()))
print(f"linhas na cronologia: {len(frame):,}, de {frame[DATE].iloc[0]} a {frame[DATE].iloc[-1]}")

linhas excluídas por não terem data: 65
linhas na cronologia: 6,469, de 2015-07-11 a 2025-02-15


In [3]:
# Passo 2 - que pergunta faz o test.csv? Só a estrutura (datas e clientes); nenhum valor
# do teste é usado para ajustar nada. As datas comparam-se como texto, como na EDA.
test_csv = pd.read_csv("../data/test.csv", index_col="ID")
test_rows_seen = test_csv[GROUP].isin(raw[GROUP])

pd.Series({
    "datas do treino": f"{frame[DATE].iloc[0]} a {frame[DATE].iloc[-1]}",
    "datas do teste": f"{test_csv[DATE].min()} a {test_csv[DATE].max()}",
    "clientes do teste já vistos no treino":
        f"{pd.Series(test_csv[GROUP].unique()).isin(raw[GROUP]).sum()} de {test_csv[GROUP].nunique()}",
    "linhas do teste com cliente já visto":
        f"{test_rows_seen.sum()} de {len(test_csv)} ({100 * test_rows_seen.mean():.1f}%)",
}).to_frame("o que o teste pergunta")

,o que o teste pergunta
datas do treino,2015-07-11 a 2025-02-15
datas do teste,2025-05-24 a 2026-10-06
clientes do teste já vistos no treino,184 de 206
linhas do teste com cliente já visto,932 de 986 (94.5%)


In [4]:
# Passo 3 - divisões por tempo: cada divisão treina no passado e valida no que vem a seguir,
# como o test.csv, que tem compras posteriores às do train.csv
tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(frame))

for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"divisão {fold}: treino até {frame[DATE].iloc[train_idx].max()}"
          f" ({len(train_idx):,} linhas), valida de {frame[DATE].iloc[val_idx].min()}"
          f" a {frame[DATE].iloc[val_idx].max()} ({len(val_idx):,} linhas)")

divisão 1: treino até 2017-08-12 (1,079 linhas), valida de 2017-08-12 a 2019-01-12 (1,078 linhas)
divisão 2: treino até 2019-01-12 (2,157 linhas), valida de 2019-01-12 a 2020-10-03 (1,078 linhas)
divisão 3: treino até 2020-10-03 (3,235 linhas), valida de 2020-10-03 a 2022-03-05 (1,078 linhas)
divisão 4: treino até 2022-03-05 (4,313 linhas), valida de 2022-03-05 a 2023-09-25 (1,078 linhas)
divisão 5: treino até 2023-09-25 (5,391 linhas), valida de 2023-09-25 a 2025-02-15 (1,078 linhas)


In [5]:
# Passo 4 - funções: preparação da linha de base, avaliação e comparação
def model_columns(frame):
    """Colunas que o modelo vê: numéricas e categóricas (sem alvo, cliente e data)."""
    features = [c for c in frame.columns if c not in OUT_OF_MODEL + [TARGET]]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(frame[c])]
    categorical = [c for c in features if c not in numeric]
    return numeric, categorical


def as_object(block):
    """O scikit-learn lê np.nan como nulo, mas não pd.NA: tudo passa a np.nan (W3)."""
    return block.astype(object).fillna(np.nan).infer_objects()


def prepare_base(train, test):
    """Linha de base: mediana nas numéricas e moda nas categóricas, ajustadas só no treino."""
    train, test = train.copy(), test.copy()
    numeric, categorical = model_columns(train)
    imputer = SimpleImputer(strategy="median").fit(train[numeric])
    train[numeric], test[numeric] = imputer.transform(train[numeric]), imputer.transform(test[numeric])
    imputer = SimpleImputer(strategy="most_frequent").fit(as_object(train[categorical]))
    train[categorical] = imputer.transform(as_object(train[categorical]))
    test[categorical] = imputer.transform(as_object(test[categorical]))
    return train, test


def fit_predict(prepare, train, test, scaler=StandardScaler()):
    """Prepara, treina no `train` e devolve as previsões (0 ou 1) para o `test`.

    Tudo é ajustado só no `train`: `prepare`, o one-hot e o scaler.
    `scaler=None` não escala. O `test` não precisa de ter o alvo.
    """
    train, test = prepare(train, test)
    numeric, categorical = model_columns(train)
    # handle_unknown="ignore": um nível que só aparece no test fica a zeros
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(as_object(train[categorical]))
    a = np.hstack([train[numeric], encoder.transform(as_object(train[categorical]))])
    b = np.hstack([test[numeric], encoder.transform(as_object(test[categorical]))])
    if scaler is not None:
        scaler.fit(a)
        a, b = scaler.transform(a), scaler.transform(b)
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(a, train[TARGET])
    return model.predict(b)


def evaluate(prepare, frame, splits, scaler=StandardScaler()):
    """F1 e accuracy de uma alternativa `prepare(train, test)` em cada divisão.

    Em cada divisão o modelo treina no treino e prevê a validação (fit_predict).
    Como o `held_out_f1` da W3.
    """
    rows = []
    for train_idx, val_idx in splits:
        train, test = frame.iloc[train_idx], frame.iloc[val_idx]
        predicted = fit_predict(prepare, train, test, scaler)
        rows.append({"f1": f1_score(test[TARGET], predicted),
                     "accuracy": accuracy_score(test[TARGET], predicted)})
    return pd.DataFrame(rows)


def sem(values):
    """Erro-padrão da média: quanto a média mudaria com outras divisões (W3)."""
    return values.std(ddof=1) / np.sqrt(len(values))


def compare(runs, reference, metric=METRIC):
    """Cada alternativa contra a linha de base, nas mesmas divisões (como na W3).

    Uma diferença menor do que 2 x SEM é ruído.
    """
    table = {}
    for name, run in runs.items():
        difference = run[metric] - reference[metric]
        table[name] = {metric: run[metric].mean(),
                       "accuracy": run["accuracy"].mean(),
                       "vs base": difference.mean(),
                       "SEM": sem(difference),
                       "melhor em": f"{int((difference > 0).sum())} de {len(difference)}"}
    return pd.DataFrame(table).T

In [6]:
# Passo 5 - linha de base: mediana / moda, one-hot, StandardScaler, sem features novas
baseline = evaluate(prepare_base, frame, splits)

print(f"linha de base: F1 {baseline['f1'].mean():.4f} +- {sem(baseline['f1']):.4f}"
      f" | accuracy {baseline['accuracy'].mean():.4f} +- {sem(baseline['accuracy']):.4f}")
baseline.round(4)

linha de base: F1 0.7321 +- 0.0090 | accuracy 0.7056 +- 0.0035


,f1,accuracy
0,0.7543,0.6929
1,0.7258,0.7106
2,0.7350,0.7124
3,0.7015,0.7032
4,0.7439,0.7087


In [7]:
# Passo 6 - o piso: prever sempre 1 (a classe maioritária) nas mesmas linhas de validação
always_one = pd.DataFrame({
    "f1": [f1_score(frame[TARGET].iloc[v], np.ones(len(v), dtype=int)) for _, v in splits],
    "accuracy": [frame[TARGET].iloc[v].mean() for _, v in splits],
})

pd.DataFrame({"linha de base": baseline.mean(), "prever sempre 1": always_one.mean()}).round(4)

,linha de base,prever sempre 1
f1,0.7321,0.7206
accuracy,0.7056,0.5638


In [8]:
# ===== [2.0] BASE COMUM - fim =====
# Como comparar uma alternativa daqui em diante (pacotes 2.1 a 2.6):
#
#   def prepare_a(train, test):      # ajusta tudo só em `train`; devolve (train, test) sem nulos
#       ...
#   compare({"alternativa A": evaluate(prepare_a, frame, splits)}, baseline)
#
# Para comparar escalas (2.4): evaluate(prepare_base, frame, splits, scaler=None) ou outro scaler.

In [9]:
# ===== [2.1] VALORES IMPOSSÍVEIS - início =====
# Passo 7 - regras de domínio da EDA (passos 9 a 11): True onde o valor é impossível
invalid_errors = {
    "days_since_previous_purchase": lambda values: values < 0,          # dias negativos (-4)
    "prior_mean_gap_days": lambda values: values < 0,                   # dias negativos (-4)
    "prior_mean_purchase_value": lambda values: (values < 0) | (values == 1e9),
    "spend_365d": lambda values: (values < 0) | (values == 1e9),
}


def invalid_flags(frame):
    """Uma coluna por regra, True onde o valor é impossível (como o domain_flags da W3).

    Mais a regra de duas colunas da EDA: uma média de compras não pode ser maior
    do que a maior dessas compras.
    """
    flags = pd.concat([rule(frame[c]) for c, rule in invalid_errors.items()], axis=1)
    flags["prior_mean_purchase_value"] |= (frame["prior_mean_purchase_value"]
                                           > frame["prior_max_purchase_value"] + 0.02)
    return flags


def blank_invalid(frame):
    """Os valores impossíveis passam a NaN (W3: "blank the cell").

    Não aprende nada com os dados, por isso aplica-se igual ao treino, à validação e ao teste.
    """
    out = frame.copy()
    out[list(invalid_errors)] = out[list(invalid_errors)].mask(invalid_flags(frame))
    return out


flags = invalid_flags(frame)
print("valores impossíveis por coluna:", flags.sum().to_dict())
print("linhas com algum valor impossível:", int(flags.any(axis=1).sum()), "de", len(frame))

valores impossíveis por coluna: {'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16, 'prior_mean_purchase_value': 77, 'spend_365d': 30}
linhas com algum valor impossível: 139 de 6469


In [10]:
# Passo 8 - manter, passar a nulo ou apagar a linha? (W3, células 63 a 69)
# Nos três casos os nulos são depois preenchidos como na linha de base (mediana / moda).
def prepare_blank(train, test):
    """Valores impossíveis passam a nulo, nas duas partes."""
    return prepare_base(blank_invalid(train), blank_invalid(test))


def prepare_drop(train, test):
    """Apaga do TREINO as linhas com valores impossíveis (W3: drop_invalid_rows).

    A validação nunca perde linhas, porque todas as compras do teste precisam de previsão.
    """
    keep = ~invalid_flags(train).any(axis=1)
    return prepare_base(train[keep], blank_invalid(test))


invalid_runs = {"passar a nulo": evaluate(prepare_blank, frame, splits),
                "apagar a linha (treino)": evaluate(prepare_drop, frame, splits)}
compare(invalid_runs, baseline).round(4)

,f1,accuracy,vs base,SEM,melhor em
passar a nulo,0.734164,0.707607,0.00206,0.0011,4 de 5
apagar a linha (treino),0.734751,0.707421,0.002646,0.001175,5 de 5


In [11]:
# ===== [2.1] VALORES IMPOSSÍVEIS - fim =====
# Passo 9 - o que fica decidido no 2.1:
#   - valores impossíveis -> passar a nulo (blank_invalid); os nulos são tratados no 2.2
#   - as 65 linhas vazias (só cliente e alvo) ficam fora do treino; já estão fora das
#     divisões desde o passo 1, e o test.csv não tem linhas assim
print("linhas do teste sem data:", int(test_csv[DATE].isna().sum()))
print("valores impossíveis no teste (a regra aplica-se igual):",
      int(invalid_flags(test_csv).sum().sum()))

linhas do teste sem data: 0
valores impossíveis no teste (a regra aplica-se igual): 18


In [12]:
# ===== [2.2] VALORES EM FALTA - início =====
# Passo 10 - nulos que sobram depois do 2.1
after_21 = blank_invalid(frame)
nulls = after_21.isna().sum()
nulls[nulls > 0].sort_values(ascending=False).to_frame("nulos")

,nulos
prior_std_gap_days,1413
prior_mean_gap_days,1259
prior_std_purchase_value,1048
prior_mean_purchase_value,874
days_since_previous_purchase,812
prior_min_purchase_value,603
prior_max_purchase_value,603
spend_365d,225
transport_method,196
payment_type,195


In [13]:
# Passo 11 - os nulos estruturais (EDA, passo 12): o valor ainda não pode existir porque
# o cliente tem poucas compras anteriores. Mínimo de compras anteriores para cada coluna existir:
needs_history = {
    "days_since_previous_purchase": 1,
    "prior_mean_purchase_value": 1,
    "prior_min_purchase_value": 1,
    "prior_max_purchase_value": 1,
    "prior_std_purchase_value": 2,   # não há desvio-padrão de um só valor
    "prior_mean_gap_days": 2,        # um intervalo precisa de duas compras
    "prior_std_gap_days": 3,         # o desvio dos intervalos precisa de dois intervalos
}


def zero_where_none(frame):
    """Zero onde o valor ainda não existe; os outros nulos ficam para a mediana.

    É o plano "zero where none / median elsewhere" da W3 (célula 79). Não aprende
    nada com os dados, por isso aplica-se igual ao treino e ao teste.
    """
    out = frame.copy()
    for column, minimum in needs_history.items():
        no_history = out[column].isna() & (out["prior_purchase_count"] < minimum)
        out.loc[no_history, column] = 0
    return out


def own_level(frame):
    """Nulos das categóricas passam a um nível próprio, "(missing)" (W3, célula 79)."""
    out = frame.copy()
    _, categorical = model_columns(out)
    out[categorical] = out[categorical].fillna("(missing)")
    return out


structural = int(after_21.isna().sum().sum() - zero_where_none(after_21).isna().sum().sum())
print(f"nulos no total: {int(after_21.isna().sum().sum()):,}, dos quais estruturais: {structural:,}")

nulos no total: 7,617, dos quais estruturais: 5,921


In [14]:
# Passo 12 - comparar planos de preenchimento (W3, células 79 a 87), todos depois do 2.1
def prepare_zero_where_none(train, test):
    return prepare_base(zero_where_none(blank_invalid(train)), zero_where_none(blank_invalid(test)))


def prepare_own_level(train, test):
    return prepare_base(own_level(blank_invalid(train)), own_level(blank_invalid(test)))


fill_runs = {"mediana / moda": invalid_runs["passar a nulo"],
             "zero sem histórico, mediana no resto": evaluate(prepare_zero_where_none, frame, splits),
             "mediana / nível próprio": evaluate(prepare_own_level, frame, splits)}
compare(fill_runs, invalid_runs["passar a nulo"]).round(4)

,f1,accuracy,vs base,SEM,melhor em
mediana / moda,0.734164,0.707607,0.0,0.0,0 de 5
"zero sem histórico, mediana no resto",0.734237,0.707421,0.000074,0.000404,2 de 5
mediana / nível próprio,0.734468,0.707607,0.000304,0.001256,2 de 5


In [15]:
# ===== [2.2] VALORES EM FALTA - fim =====
# Passo 13 - o que fica decidido no 2.1 e no 2.2, numa só preparação para os pacotes seguintes:
#   1. valores impossíveis -> nulo (blank_invalid), igual no treino e no teste
#   2. nulos -> mediana nas numéricas e moda nas categóricas, ajustadas só no treino
#      (as alternativas ficaram dentro do ruído; como na W3, fica a mais simples)
prepare_clean = prepare_blank

clean = invalid_runs["passar a nulo"]
print(f"2.1 + 2.2: F1 {clean['f1'].mean():.4f} +- {sem(clean['f1']):.4f}"
      f" | accuracy {clean['accuracy'].mean():.4f} +- {sem(clean['accuracy']):.4f}")

2.1 + 2.2: F1 0.7342 +- 0.0093 | accuracy 0.7076 +- 0.0024


In [16]:
# ===== [2.4] DISTRIBUIÇÕES E ESCALA - início =====
# Passo 1 - que colunas transformar? Todas as numéricas com cauda longa (EDA, passo 7):
# todas menos random_noise e customer_tenure_days, as duas únicas sem cauda longa.
# Os dois scalers novos importam-se aqui para não mexer na célula de imports (juntar no 2.7).
from sklearn.preprocessing import MinMaxScaler, RobustScaler

skewed = [c for c in model_columns(frame)[0] if c not in ["random_noise", "customer_tenure_days"]]
print(f"colunas a transformar: {len(skewed)} de {len(model_columns(frame)[0])} numéricas")

# Assimetria (skew) antes e depois do log1p, depois de tirar os valores impossíveis (2.1).
# Só para ver o efeito: aqui nada é ajustado, por isso pode usar o treino inteiro (W3, célula 28).
after_21 = blank_invalid(frame)
pd.DataFrame({"skew antes": after_21[skewed].skew(),
              "skew depois do log1p": np.log1p(after_21[skewed]).skew()}).round(2)

colunas a transformar: 27 de 29 numéricas


,skew antes,skew depois do log1p
Total a Pagar,4.21,-0.48
Total Bruto,4.31,-0.43
Total Liquido,4.15,-0.45
Total Imp_,4.84,-0.76
Total Portes,19.55,2.27
Total Desc_ Global,27.46,9.28
Total Desc_ Linha,5.92,0.24
Custo,4.61,-1.02
N_ Detalhes,2.98,0.08
invoice_count_current_purchase,6.44,5.00


In [17]:
# Passo 2 - os dois tratamentos, sempre pela ordem: valores impossíveis -> tratamento -> nulos.
# Depois do blank_invalid já não há negativos nestas colunas, por isso o log1p não precisa
# do clip(lower=0) que a W3 usa no log1p_of.
def prepare_log1p(train, test):
    """log1p nas colunas com cauda longa (W3, célula 100: log1p_of).

    Não aprende nada com os dados, por isso aplica-se igual ao treino e ao teste.
    """
    train, test = blank_invalid(train), blank_invalid(test)
    train[skewed], test[skewed] = np.log1p(train[skewed]), np.log1p(test[skewed])
    return prepare_base(train, test)


def prepare_winsor(train, test):
    """Corta cada coluna aos percentis 1% e 99% do TREINO (W3, célula 100: winsorize).

    Os limites aprendem-se só no `train` e aplicam-se iguais ao `test`.
    """
    train, test = blank_invalid(train), blank_invalid(test)
    low, high = train[skewed].quantile(0.01), train[skewed].quantile(0.99)
    train[skewed] = train[skewed].clip(low, high, axis=1)
    test[skewed] = test[skewed].clip(low, high, axis=1)
    return prepare_base(train, test)

In [18]:
# Passo 3 - comparar os tratamentos com a referência `clean` (2.1 + 2.2), nas mesmas divisões
# e com o mesmo StandardScaler (W3, células 100 a 104). Só conta se "vs base" > 2 x SEM.
treatment_runs = {"log1p": evaluate(prepare_log1p, frame, splits),
                  "winsorize 1% / 99%": evaluate(prepare_winsor, frame, splits)}
treatment_table = compare(treatment_runs, clean)
treatment_table["2 x SEM"] = 2 * treatment_table["SEM"]
treatment_table.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
log1p,0.739642,0.707236,0.005478,0.002904,4 de 5,0.005808
winsorize 1% / 99%,0.737703,0.709647,0.003539,0.002745,3 de 5,0.005491


In [19]:
# Passo 4 - comparar escalas com a melhor média do passo 3, o log1p (W3, células 112 a 114).
# A referência continua a ser `clean`, para todas as linhas se lerem contra o mesmo número.
# Sem scaler, a regressão logística pára no limite de 1000 iterações sem convergir: daí os
# avisos ConvergenceWarning em baixo (a W3 mostra o mesmo).
scaler_runs = {"log1p + StandardScaler": treatment_runs["log1p"],
               "log1p + MinMaxScaler": evaluate(prepare_log1p, frame, splits, scaler=MinMaxScaler()),
               "log1p + RobustScaler": evaluate(prepare_log1p, frame, splits, scaler=RobustScaler()),
               "log1p sem scaler": evaluate(prepare_log1p, frame, splits, scaler=None)}
scaler_table = compare(scaler_runs, clean)
scaler_table["2 x SEM"] = 2 * scaler_table["SEM"]
scaler_table.round(4)

c:\Users\Acer\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Acer\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/st

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
log1p + StandardScaler,0.739642,0.707236,0.005478,0.002904,4 de 5,0.005808
log1p + MinMaxScaler,0.748076,0.71577,0.013912,0.002819,5 de 5,0.005637
log1p + RobustScaler,0.743428,0.713173,0.009264,0.002626,5 de 5,0.005251
log1p sem scaler,0.743537,0.71039,0.009374,0.003582,4 de 5,0.007164


In [20]:
# ===== [2.4] DISTRIBUIÇÕES E ESCALA - fim =====
# Passo 5 - o que fica decidido no 2.4 (números contra `clean`, F1 0.7342):
#   1. tratamento -> log1p nas 27 colunas com cauda longa
#      - sozinho (com StandardScaler) fica no limite do ruído: +0.0055, com 2 x SEM = 0.0058;
#        o winsorize fica abaixo (+0.0035, 2 x SEM = 0.0055)
#      - o log1p não aprende nada com os dados (não há fuga) e é o tratamento da W3
#   2. escala -> MinMaxScaler
#      - log1p + MinMaxScaler: F1 0.7481, +0.0139 com 2 x SEM = 0.0056, melhor em 5 de 5
#        divisões: é a única diferença claramente acima do ruído
#      - RobustScaler também passa os 2 x SEM (+0.0093 contra 0.0053), mas fica abaixo
#      - sem scaler o modelo não converge, por isso fica de fora mesmo com boa média
#   Nota: o scaler não faz parte do `prepare`; passa-se a evaluate / fit_predict com
#   scaler=SCALER_24. A célula de submissão ainda usa a preparação do 2.2 (juntar no 2.7).
prepare_24 = prepare_log1p
SCALER_24 = MinMaxScaler()

result_24 = scaler_runs["log1p + MinMaxScaler"]
print(f"2.4: F1 {result_24['f1'].mean():.4f} +- {sem(result_24['f1']):.4f}"
      f" | accuracy {result_24['accuracy'].mean():.4f} +- {sem(result_24['accuracy']):.4f}")

2.4: F1 0.7481 +- 0.0086 | accuracy 0.7158 +- 0.0052


In [21]:
# ===== [2.6] NOVAS FEATURES - início =====
# Passo 1 - as features candidatas. Todas usam só informação que existe no momento da compra.
# Nas divisões, um denominador zero passa a nulo (W3, célula 28: .replace(0, np.nan));
# esse nulo é depois preenchido pela mediana do treino no prepare_base.
new_features = {
    # a primeira compra comporta-se de outra forma (EDA, passo 12: 33% contra 58% de recompra)
    "is_first_purchase": lambda t: (t["prior_purchase_count"] == 0).astype(int),
    # atraso face ao intervalo habitual do cliente
    "days_vs_habit": lambda t: t["days_since_previous_purchase"]
                               / t["prior_mean_gap_days"].replace(0, np.nan),
    # valor médio por compra no último ano
    "avg_ticket_365d": lambda t: t["spend_365d"] / t["purchase_count_365d"].replace(0, np.nan),
    # margem da compra
    "margin": lambda t: t["Total Liquido"] - t["Custo"],
    # peso do desconto na compra
    "discount_share": lambda t: t["Total Desc_ Linha"] / t["Total Bruto"].replace(0, np.nan),
}


def add_features(frame, names=tuple(new_features)):
    """Acrescenta as features `names` a uma tabela. Não aprende nada com os dados."""
    out = frame.copy()
    for name in names:
        out[name] = new_features[name](out)
    return out


after_26 = add_features(blank_invalid(frame))
after_26[list(new_features)].describe().T[["count", "min", "50%", "max"]].round(3)

,count,min,50%,max
is_first_purchase,6469.0,0.000,0.000,1.000
days_vs_habit,5031.0,0.001,0.831,478.667
avg_ticket_365d,5434.0,3.310,720.128,66539.157
margin,6469.0,-1592.119,197.205,11075.600
discount_share,6469.0,0.000,0.046,0.975


In [22]:
# Passo 2 - uma preparação para cada conjunto de features, sempre pela ordem
# blank_invalid -> add_features -> prepare_base: as features calculam-se antes de preencher
# os nulos, para não usarem valores inventados.
def prepare_with(names):
    """Devolve uma função prepare(train, test) = prepare_clean + as features `names`."""
    def prepare(train, test):
        train = add_features(blank_invalid(train), names)
        test = add_features(blank_invalid(test), names)
        return prepare_base(train, test)
    return prepare

In [23]:
# Passo 3 - cada feature sozinha, acrescentada a prepare_clean, contra `clean`
# (W4, "Combination features", passos 9 e 10: paired_delta). Só conta se "vs base" > 2 x SEM.
# A última linha junta as cinco, só para ver se juntas fazem o que sozinhas não fazem.
feature_runs = {name: evaluate(prepare_with([name]), frame, splits) for name in new_features}
feature_runs["todas as 5"] = evaluate(prepare_with(list(new_features)), frame, splits)
feature_table = compare(feature_runs, clean)
feature_table["2 x SEM"] = 2 * feature_table["SEM"]
feature_table.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
is_first_purchase,0.734286,0.707607,0.000122,0.00075,1 de 5,0.0015
days_vs_habit,0.734271,0.707421,0.000107,0.00069,2 de 5,0.001379
avg_ticket_365d,0.733388,0.706865,-0.000776,0.000222,0 de 5,0.000444
margin,0.734217,0.707792,0.000053,0.000306,2 de 5,0.000611
discount_share,0.736052,0.709091,0.001888,0.001013,4 de 5,0.002027
todas as 5,0.736541,0.708349,0.002378,0.001351,4 de 5,0.002701


In [24]:
# Passo 4 - confirmação (não pedida no enunciado do 2.6): as mesmas features por cima da
# receita do 2.4 (log1p + MinMaxScaler), contra `result_24`. Ajuda a decidir no 2.7.
def prepare_with_24(names):
    """prepare_24 + as features `names`; o log1p só mexe nas colunas originais (`skewed`)."""
    def prepare(train, test):
        train = add_features(blank_invalid(train), names)
        test = add_features(blank_invalid(test), names)
        train[skewed], test[skewed] = np.log1p(train[skewed]), np.log1p(test[skewed])
        return prepare_base(train, test)
    return prepare


feature_runs_24 = {name: evaluate(prepare_with_24([name]), frame, splits, scaler=SCALER_24)
                   for name in new_features}
feature_table_24 = compare(feature_runs_24, result_24)
feature_table_24["2 x SEM"] = 2 * feature_table_24["SEM"]
feature_table_24.round(4)

,f1,accuracy,vs base,SEM,melhor em,2 x SEM
is_first_purchase,0.746513,0.714286,-0.001563,0.000773,1 de 5,0.001546
days_vs_habit,0.748472,0.716327,0.000396,0.000597,2 de 5,0.001194
avg_ticket_365d,0.748189,0.715955,0.000113,0.000489,2 de 5,0.000978
margin,0.748568,0.716512,0.000492,0.000282,3 de 5,0.000565
discount_share,0.747299,0.715213,-0.000777,0.00083,2 de 5,0.00166


In [25]:
# ===== [2.6] NOVAS FEATURES - fim =====
# Passo 5 - o que fica decidido no 2.6 (números contra `clean`, F1 0.7342):
#   - nenhuma feature passa os 2 x SEM, por isso nenhuma fica:
#       is_first_purchase +0.0001, 2 x SEM = 0.0015, melhor em 1 de 5
#       days_vs_habit     +0.0001, 2 x SEM = 0.0014, melhor em 2 de 5
#       avg_ticket_365d   -0.0008, 2 x SEM = 0.0004, melhor em 0 de 5   <- pior em todas as divisões
#       margin            +0.0001, 2 x SEM = 0.0006, melhor em 2 de 5
#       discount_share    +0.0019, 2 x SEM = 0.0020, melhor em 4 de 5   <- a mais perto, mas ainda dentro do ruído
#   - o passo "testar juntas as que ganham" fica vazio; mesmo as cinco juntas ficam no ruído
#     (+0.0024, 2 x SEM = 0.0027, melhor em 4 de 5)
#   - por cima do 2.4 (passo 4) a conclusão é a mesma: nenhuma passa os 2 x SEM
#   - porquê: a informação já está nas colunas que existem (por exemplo, is_first_purchase
#     é só prior_purchase_count == 0, e prior_purchase_count já entra no modelo)
#   Como as alternativas empatam, fica a mais simples: sem features novas.
prepare_26 = prepare_clean

In [26]:
# ===== SUBMISSÃO AO KAGGLE =====
# Passo 14 - a preparação escolhida (2.1 + 2.2) e o modelo, ajustados no treino inteiro,
# preveem as 986 compras do test.csv. Formato do sample_submission: ID e previsão (0 ou 1).
TEAM = "ML_Group_XX"   # trocar XX pelo número do grupo
VERSION = 1            # aumentar a cada nova submissão

predictions = fit_predict(prepare_clean, frame, test_csv)
submission = pd.DataFrame({"ID": test_csv.index, TARGET: predictions})

path = f"../submissions/{TEAM}_Version{VERSION:02d}.csv"
submission.to_csv(path, index=False)   # sem o índice do pandas, como pede o guia do Kaggle

sample = pd.read_csv("../data/sample_submission.csv")
print(f"gravado em {path}")
print(f"linhas: {len(submission)}, IDs únicos: {submission['ID'].nunique()},"
      f" mesmos IDs do sample_submission: {submission['ID'].isin(sample['ID']).all()}")
print("previsões:", submission[TARGET].value_counts().sort_index().to_dict())

gravado em ../submissions/ML_Group_XX_Version01.csv
linhas: 986, IDs únicos: 986, mesmos IDs do sample_submission: True
previsões: {0: 401, 1: 585}
